# [예제1 네이버 검색하기](https://tkdrms568.tistory.com/184)

In [1]:
import time 
from selenium import webdriver 
from selenium.webdriver.common.by import By 
from selenium.webdriver import Keys, ActionChains
import pyperclip 

In [2]:
import platform 
os_base = platform.system()
os_base

'Windows'

In [3]:
driver = webdriver.Chrome() 
url = "http://www.naver.com"
driver.get(url)
driver.implicitly_wait(3)

In [4]:
search_query = "아이유"

query_id = driver.find_element(By.ID, value="query")
query_id.click()
pyperclip.copy(search_query)
driver.implicitly_wait(3)

if os_base == 'Darwin':
    query_id.send_keys(Keys.COMMAND, 'v') # 맥북인 경우 
else:
    query_id.send_keys(Keys.CONTROL, 'v') # 윈도우인 경우
    
driver.implicitly_wait(3)

In [5]:
# 네이버 UI 개편으로 검색 버튼(.btn_search)이 화면에 보이지 않아(display: none) 클릭이 불가능해짐
# -> 검색창에서 Enter 키를 입력해 폼을 제출
query_id.send_keys(Keys.RETURN)
driver.implicitly_wait(3)

# [고시공고 조회 예제](https://medium.com/@crownsy4/python-%EB%8F%99%EC%A0%81%ED%8E%98%EC%9D%B4%EC%A7%80-%ED%81%AC%EB%A1%A4%EB%A7%81-7a0ed5433c4b)

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException

In [7]:
def find_page_link(driver, page_number):
    """페이지 번호 링크를 찾는다. 없으면 '다음' 그룹으로 넘어가서 다시 찾는다."""

    try:
        return driver.find_element(By.XPATH, f'//ul[@class="pagination"]/li/a[text()="{page_number}"]')
    except NoSuchElementException:
        # 현재 보이는 페이지 번호 그룹(1~10, 11~20 ...)에 없으면
        # '다음' 버튼을 눌러 다음 그룹으로 이동한 뒤 다시 시도
        next_page_link = driver.find_element(By.XPATH, '//ul[@class="pagination"]/li/a[@aria-label="Next"]')
        next_page_link.click()
        return driver.find_element(By.XPATH, f'//ul[@class="pagination"]/li/a[text()="{page_number}"]')


def crawl_and_insert_data(url):

    # Chrome 드라이버 생성
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Chrome을 headless 모드로 실행

    driver = webdriver.Chrome(options=chrome_options)

    # 페이지 로딩이 완료될 때까지 대기하는 코드
    driver.implicitly_wait(5)

    # 사이트 접속
    driver.get(url)

    # 마지막 페이지 번호 가져오기
    page_links = driver.find_elements(By.XPATH, '//ul[@class="pagination"]/li/a')
    last_page = 0
    for link in page_links:
        try:
            if link.get_attribute("aria-label") == "last":
                href = link.get_attribute("href")
                page_index = int(href.split("=")[-1])
                last_page = page_index
        except ValueError:
            pass

    # 마지막 페이지 확인
    print("last_page: "+str(last_page))

    for page_number in range(1, last_page + 1):
        try:
            # 페이지 번호 클릭
            page_link = find_page_link(driver, page_number)
            page_link.click()

            values = []  # values 변수 초기화

            # 각 페이지의 모든 tr 태그 선택
            tbody_tag = driver.find_element(By.TAG_NAME, "tbody")
            tr_tags = tbody_tag.find_elements(By.TAG_NAME, "tr")
            
            # 각 tr 태그에 대해 반복하여 td 태그의 텍스트 가져오기
            for tr_tag in tr_tags:
                td_tags = tr_tag.find_elements(By.TAG_NAME, "td")
                cnt = 0
                for td_tag in td_tags:
                    # td 태그의 텍스트 출력
                    text = td_tag.text.strip()
                    # if(cnt == 0) :
                    #   text = int(text)  
                    if("\n새글" in text):
                        text = text.replace("\n새글","")

                    values.append(text)
                    cnt = cnt + 1
                print("======================================")
                print("page_number: "+str(page_number))
                print(values) 

                values = []  # 값 추가 후에 values 초기화

            if page_number >= 21:
                break # 21까지만 크롤링!!

        except StaleElementReferenceException:
            # 요소가 만료되었을 때 다시 요소를 찾아서 시도
            page_link = find_page_link(driver, page_number)
            page_link.click()
        except NoSuchElementException as e:
            # 재시도로도 페이지 번호를 찾지 못하면 남은 페이지 크롤링을 중단
            print(f"page_number {page_number}에 대한 페이지 링크를 찾지 못해 중단합니다: {e}")
            break

    # Selenium 사용 종료
    driver.quit()

In [8]:
crawl_and_insert_data('https://www.yuseong.go.kr/prog/saeolGosi/GOSI/kor/sub04_02_01/list.do') #유성구청 고시공고

last_page: 96
page_number: 1
['957', '대전광역시 유성구 공고 제2026-1643호', '2026년 8월 자동차관리법 위반 원상복구, 정비명령서 반송에 따른 공시송달 공고', '교통정책과', '', '2026-08-21 ~ 2026-09-07']
page_number: 1
['956', '대전광역시 유성구 공고 제2026-1642호', '주정차위반과태료 사전통지서 선택등기우편 반송분 공시송달 공고', '주차관리과', '', '2026-08-21 ~ 2026-09-04']
page_number: 1
['955', '대전광역시 유성구 공고 제2026-1641호', '건설업 행정처분(등록말소) 공고', '건설과', '', '2026-08-21 ~ 2026-09-07']
page_number: 1
['954', '대전광역시 유성구 공고 제2026-1640호', '담배소매인 폐업에 따른 지정신청 공고(봉산동)', '일자리정책과', '', '2026-08-21 ~ 2026-08-28']
page_number: 1
['953', '대전광역시 유성구 온천1동 공고 제2026-74호', '신규 주민등록증 발급통지서 반송자 공고(2009년 8월생)', '온천1동', '', '2026-08-21 ~ 2026-09-04']
page_number: 1
['952', '대전광역시 유성구 공고 제2026-1637호', '담배소매인 폐업에 따른 지정신청 공고(용산동)', '일자리정책과', '', '2026-08-21 ~ 2026-08-28']
page_number: 1
['951', '대전광역시 유성구 고시 제2026-108호', '지적기준점(지적도근점) 성과고시', '토지정보과', '', '']
page_number: 1
['950', '대전광역시 유성구 보건소 공고 제2026-29호', '감염병예방법 위반 소독업소 행정처분 사전통지 공시송달 공고', '보건의약과', '', '2026-08-21 ~ 2026-09-04']
page_number: 1
['949

In [9]:
crawl_and_insert_data('https://www.djjunggu.go.kr/prog/saeolGosi/GOSI/sub03_06/list.do')#중구청 고시공고

last_page: 1747
page_number: 1
['17462', '대전광역시 중구 공고 제2026-1252호', '소유자 미상 방치 이륜자동차 자진처리 안내 공고', '교통행정과', '2026-08-21', '2026-08-21 ~ 2026-09-20']
page_number: 1
['17461', '대전광역시 중구 대흥동 공고 제2026-39호', '거주불명등록자 행정상관리주소이전 직권조치 결과 공고', '대흥동', '2026-08-21', '2026-08-21 ~ 2026-09-10']
page_number: 1
['17460', '대전광역시 중구 공고 제2026-1251호', '「자동차관리법」위반(이륜차검사지연)에 따른 과태료 본부과 반송분 공시송달 공고', '교통행정과', '2026-08-21', '2026-08-21 ~ 2026-09-04']
page_number: 1
['17459', '대전광역시 중구 공고 제2026-1250호', '「자동차관리법」위반(이륜차검사지연)에 따른 과태료 독촉 및 압류예고 반송분 공시송달 공고', '교통행정과', '2026-08-21', '2026-08-21 ~ 2026-09-04']
page_number: 1
['17458', '대전광역시 중구 중촌동 공고 제2026-22호', '거주불명등록자 행정상 관리주소 이전 직권조치 결과 공고', '중촌동', '2026-08-21', '2026-08-21 ~ 2026-09-07']
page_number: 1
['17457', '대전광역시 중구 공고 제2026-1248호', '제275회 중구의회 제1차 정례회 제출 안건 공고', '기획홍보실', '2026-08-21', '2026-08-21 ~ 2026-09-21']
page_number: 1
['17456', '대전광역시 중구 공고 제2026-1246호', '담배소매업 폐업 및 지정신청 접수 공고', '일자리경제과', '2026-08-20', '2026-08-20 ~ 2026-08-27']
page_number: 1
['

# 현대자동차 FAQ 크롤링 예제

In [10]:
import re
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

In [11]:
def clean_text(text: str) -> str:
    """불필요한 공백 정리"""
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_total_pages(driver):
    """페이지네이션에서 전체 페이지 수 확인"""

    try:
        end_button = driver.find_element(By.CSS_SELECTOR, ".ui_paging button.navi.end")
        return int(end_button.get_attribute("data-page"))
    except NoSuchElementException:
        return 1


def go_to_next_page(driver):
    """다음 페이지 버튼 클릭"""

    next_button = driver.find_element(By.CSS_SELECTOR, ".ui_paging button.navi.next")
    next_button.click()

    # 목록 갱신 대기
    time.sleep(1.5)


def get_faq_items(driver):
    """현재 페이지에 로드된 FAQ 목록 수집"""

    items = []

    faq_list = driver.find_elements(By.CSS_SELECTOR, ".result_area .ui_accordion dl")

    for dl in faq_list:

        # -------------------------
        # 카테고리 (접힌 항목은 화면에 보이지 않으므로 textContent로 추출)
        # -------------------------
        try:
            category = clean_text(
                dl.find_element(By.CSS_SELECTOR, "dt .title i").get_attribute("textContent")
            ).strip("[] ")
        except NoSuchElementException:
            category = ""

        # -------------------------
        # 질문
        # -------------------------
        try:
            question = clean_text(
                dl.find_element(By.CSS_SELECTOR, "dt .title .brief").text
            )
        except NoSuchElementException:
            question = ""

        # -------------------------
        # 답변 (접힌 항목은 화면에 보이지 않으므로 textContent로 추출)
        # -------------------------
        try:
            answer = clean_text(
                dl.find_element(By.CSS_SELECTOR, "dd .exp").get_attribute("textContent")
            )
        except NoSuchElementException:
            answer = ""

        items.append({
            "category": category,
            "question": question,
            "answer": answer
        })

    return items


def get_all_faqs(driver, FAQ_URL, MAX_PAGES):
    """
    현대자동차 FAQ 페이지에서
    전체 페이지를 순회하며 FAQ 목록 수집
    """

    driver.get(FAQ_URL)
    driver.implicitly_wait(5)

    # 동적 콘텐츠 로딩 대기
    time.sleep(2)

    total_pages = get_total_pages(driver)
    target_pages = min(total_pages, MAX_PAGES)

    print(f"전체 페이지: {total_pages}개 / 수집 대상: {target_pages}개")

    results = []

    for page_no in range(1, target_pages + 1):

        items = get_faq_items(driver)

        print(f"[{page_no}/{target_pages}] {len(items)}개 수집")

        results.extend(items)

        if page_no < target_pages:
            go_to_next_page(driver)

        # 너무 빠르게 요청하지 않도록 간격
        time.sleep(1)

    return results

In [12]:
FAQ_URL = "https://www.hyundai.com/kr/ko/faq.html"
MAX_PAGES = 3

driver = webdriver.Chrome()

# 1. FAQ 목록 수집
faq_results = get_all_faqs(driver, FAQ_URL, MAX_PAGES)

driver.quit()

print()
print(f"최종 수집 건수: {len(faq_results)}")
faq_results[:5]

전체 페이지: 6개 / 수집 대상: 3개
[1/3] 10개 수집
[2/3] 10개 수집
[3/3] 10개 수집

최종 수집 건수: 30


[{'category': '모젠서비스 > 이용단말',
  'question': '내비게이션의 지도 업그레이드는 어떻게 하나요?',
  'answer': '모젠 지도 업그레이드는 차량용 A/V 및 내비게이션 A/S를 담당하고 있는현대 웰슨에서 수행하고 있습니다. 전국 현대자동차 23개 직영서비스 센터와 기아자동차 11개 직영서비스 센터에서현대 웰슨 A/S요원이 상주 근무하고 있으며전국 115개소의 현대 웰슨 협력점에서 지도 업그레이드가 가능합니다. 모젠고객센터 1588-8111로 전화주시면 지도 업그레이드 상담이 가능하며업그레이드 가능한 가까운 서비스센터에 대한 안내를 받으실 수 있습니다.'},
 {'category': '블루링크 > 가입/해지/변경',
  'question': '[가입] 블루링크에 가입하려면 어떻게 해야 하나요?',
  'answer': '차종에 따른 가입방법은 현대자동차 홈페이지 내 블루링크 이용안내 페이지에서 확인하실 수 있습니다.※ 현대자동차 홈폐이지 > 고객서비스 > 블루링크 > 이용안내 > 서비스 가입 ▶바로가기'},
 {'category': '차량정비 > 일반',
  'question': '에어컨 성능을 향상시키는 방법 안내',
  'answer': '●에어컨 작동은 1단부터 작동하기 보다는 최고단수부터 작동하는 것이 좋습니다. ☞에어컨을 처음 틀때 1,2단이 아닌 3,4단부터 시작하는 게 냉각효율과 에너지 절약에 도움되며, 공기순환 모드를 잘 활용하면 냉방효과를 더욱 높일 수 있습니다. 차내 급속 냉방을 위해서는 내기순환 모드에서 에어컨을 가동하고, 최고단수로 5분 정도 작동한 뒤조정하는 게 효과적입니다. ●에어컨 콘덴서(응축기)를 청소합시다. ☞엔진오일 교환이나 세차 때 정비사에게 에어컨 컨덴서(응축기) 외부에 붙어있는 벌레,이물질,먼지 등을 압축공기나 고압세차기로 청소해달라고 부탁하면 10% 정도의 냉각효을 상승 효과를 기대할 수 있습니다. 콘덴서는 라디에이터 앞에 설치돼 차량속도와 냉각팬에 의해 기체상태의 냉매를 고압의 액체상